# Diagnosis: why are the scores so low?

The professor asked why our fine-tuned CodeBERT reaches only 0.031 macro-F1 with all training labels. There are two possible reasons. Either the project-wise split removes leakage that made earlier results look better, or something in the fine-tuning pipeline is wrong. This notebook tests this. It uses the same data, the same project split (seed 1060) and the same 82 classes as `src/analysis.ipynb`.

The notebook runs these parts:

1. The share of functions that are longer than 512 tokens (truncation rate).
2. A majority-class baseline.
3. TF-IDF with logistic regression.
4. Frozen CodeBERT with logistic regression (the Presentation 1 setup), on the project split and on a random split.
5. The reference model trained again with all labels (seed 0), with its training scores and one prediction per test function.
6. A confusion matrix and three randomly chosen misclassified test functions.
7. A summary table and the decision rule.

**Decision rule** (written in `results.md` before we ran this notebook): look at the test macro-F1 of frozen CodeBERT with logistic regression on the project split. At 0.05 or lower, the project split explains the low scores. At 0.10 or higher, the fine-tuning pipeline is probably broken. Between the two, the result is unclear.

## How to run on RunPod

1. Start a pod with a CUDA GPU (for example an RTX 4090) and a PyTorch template.
2. Upload this notebook and `requirements.txt` to the same folder, for example `/workspace/`.
3. Optional but recommended: also upload `results/project_split.csv` to `/workspace/results/project_split.csv`. The notebook then checks that the split is the same as in the paper.
4. Run all cells from top to bottom. The whole notebook takes about 45 to 60 minutes on an RTX 4090. Most of this time is the reference model in section 9.
5. Download `results/diagnosis.zip` at the end. It holds all result files and figures.

Each slow step saves its result. If the pod stops, run the notebook again: it skips the steps that are already done. BigVul and MegaVul are not needed here.

In [ ]:
%pip install -q -r requirements.txt

## 1. Libraries

This cell loads the libraries and prints their versions. We need the versions for the README.

In [ ]:
from pathlib import Path
import random
import shutil
import sys

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
import matplotlib
import matplotlib.pyplot as plt
from huggingface_hub import snapshot_download
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print("Python", sys.version.split()[0])
for lib in [np, pd, sklearn, torch, transformers, matplotlib]:
    print(lib.__name__, lib.__version__)
print("CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Setup

All new files go to `results/diagnosis/` and `figures/diagnosis/`. We do not overwrite any result of the main analysis.

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DIVERSEVUL_REPO = "claudios/DiverseVul"
DIVERSEVUL_REVISION = "98c294b3caaa94170e77282f1587ea20c3d59784"
RESULTS_DIR = ROOT / "results"
OUT_DIR = RESULTS_DIR / "diagnosis"
FIG_DIR = ROOT / "figures" / "diagnosis"
MODEL_OUT = ROOT / "models" / "diagnosis_reference"
CHECKPOINT_DIR = ROOT / "checkpoints"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

## 3. Load and clean DiverseVul

This cell is a copy of section 3 in `src/analysis.ipynb`. It should give 16,098 functions, 139 classes and 739 projects.

In [ ]:
MIN_COUNT = 2

diversevul_dir = Path(snapshot_download(DIVERSEVUL_REPO, repo_type="dataset", revision=DIVERSEVUL_REVISION))
diversevul = pd.concat(pd.read_parquet(path) for path in sorted(diversevul_dir.glob("data/*.parquet"))).reset_index(drop=True)
diversevul["cwe"] = diversevul["cwe"].map(list)

labeled = diversevul[
    (diversevul["target"] == 1)
    & (diversevul["cwe"].map(len) > 0)
    & (diversevul["func"].astype(str).str.strip() != "")
].copy()

cwe_freq = labeled["cwe"].explode().value_counts()


# We take the rarest CWE when there are several
def pick_cwe_label(cwe_list):
    return min(cwe_list, key=lambda cwe: (cwe_freq[cwe], cwe_list.index(cwe)))


labeled["cwe_label"] = labeled["cwe"].map(pick_cwe_label)
label_counts = labeled["cwe_label"].value_counts()
labeled = labeled[labeled["cwe_label"].isin(label_counts[label_counts >= MIN_COUNT].index)]

data = labeled[["func", "cwe_label", "project", "commit_id"]].reset_index(drop=True)

print("Rows:", len(data), "| classes:", data["cwe_label"].nunique(), "| projects:", data["project"].nunique())
assert (len(data), data["cwe_label"].nunique(), data["project"].nunique()) == (16098, 139, 739)

## 4. Train, validation and test split

This cell is a copy of section 4 in `src/analysis.ipynb`. If `results/project_split.csv` exists, the cell checks that every project is in the same split as in the paper. It does not overwrite that file. At the end, the train split should have 11,484 functions and 82 classes, and the test split 2,424 functions.

In [ ]:
SPLIT_SEED = 1060
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
MIN_TRAIN = 10
RARE_MAX = 50

projects = data["project"].drop_duplicates().sort_values().to_numpy()
order = np.random.RandomState(SPLIT_SEED).permutation(len(projects))
n_train = int(TRAIN_FRAC * len(projects))
n_val = int(VAL_FRAC * len(projects))

split_of_project = pd.Series("test", index=projects)
split_of_project.iloc[order[:n_train]] = "train"
split_of_project.iloc[order[n_train:n_train + n_val]] = "val"
data["split"] = data["project"].map(split_of_project)

saved_split_path = RESULTS_DIR / "project_split.csv"
if saved_split_path.exists():
    saved_split = pd.read_csv(saved_split_path, index_col="project")["split"]
    assert saved_split.sort_index().equals(split_of_project.sort_index()), "The split is not the same as in the paper"
    print("Split check: same as results/project_split.csv")
else:
    print("results/project_split.csv not found, so the split check is skipped")

train_df = data[data["split"] == "train"].reset_index(drop=True)
val_df = data[data["split"] == "val"].reset_index(drop=True)
test_df = data[data["split"] == "test"].reset_index(drop=True)

all_train_counts = train_df["cwe_label"].value_counts()
kept_classes = all_train_counts[all_train_counts >= MIN_TRAIN].index
train_df = train_df[train_df["cwe_label"].isin(kept_classes)].reset_index(drop=True)
val_df = val_df[val_df["cwe_label"].isin(kept_classes)].reset_index(drop=True)
test_df = test_df[test_df["cwe_label"].isin(kept_classes)].reset_index(drop=True)

train_counts = train_df["cwe_label"].value_counts()
rare_classes = set(train_counts[train_counts < RARE_MAX].index)

splits = {"train": train_df, "val": val_df, "test": test_df}
print(pd.DataFrame({name: {"rows": len(df), "classes": df["cwe_label"].nunique()} for name, df in splits.items()}).T)
assert (len(train_df), train_df["cwe_label"].nunique(), len(val_df), len(test_df)) == (11484, 82, 1689, 2424)

## 5. Shared code

This cell is a copy of section 5 in `src/analysis.ipynb`, with the same model, settings and class-weighted loss. We add one function, `scores`. It gives the macro-F1, the mean F1 of the rare and the common classes, and the accuracy. Macro-F1 uses the same definition as the paper: the mean F1 over the classes that have true examples in the evaluation set. A class whose F1 cannot be computed counts as 0. Every model can predict all 82 training classes.

In [ ]:
MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP = 0.1
EPOCHS = 10

classes = sorted(train_counts.index)
label2id = {label: i for i, label in enumerate(classes)}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)


class CodeDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
        self.label_ids = None if labels is None else [label2id[label] for label in labels]

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.encodings.items()}
        if self.label_ids is not None:
            item["labels"] = self.label_ids[i]
        return item


class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=sorted(set(y_true)), average="macro", zero_division=0)


def per_class_f1(y_true, y_pred):
    labels = sorted(set(y_true))
    result = pd.DataFrame({
        "cwe": labels,
        "f1": f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0),
        "train_count": train_counts[labels].to_numpy(),
    })
    result["group"] = np.where(result["cwe"].isin(rare_classes), "rare", "common")
    return result


def scores(model, split, y_true, y_pred):
    per_class = per_class_f1(list(y_true), list(y_pred))
    groups = per_class.groupby("group")["f1"].mean()
    return {
        "model": model,
        "split": split,
        "macro_f1": per_class["f1"].mean(),
        "rare_f1": groups.get("rare", np.nan),
        "common_f1": groups.get("common", np.nan),
        "accuracy": accuracy_score(list(y_true), list(y_pred)),
        "classes": len(per_class),
    }


def compute_metrics(pred):
    y_true = np.array(classes)[pred.label_ids]
    y_pred = np.array(classes)[pred.predictions.argmax(axis=1)]
    return {"macro_f1": macro_f1(y_true, y_pred)}


def train_model(texts, labels, seed, epochs=EPOCHS):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(classes))
    train_data = CodeDataset(texts, labels)
    weights = compute_class_weight("balanced", classes=np.arange(len(classes)), y=train_data.label_ids)

    args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        num_train_epochs=epochs,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        save_total_limit=1,
        save_only_model=True,
        report_to="none",
        seed=seed,
    )
    trainer = WeightedTrainer(
        class_weights=torch.tensor(weights, dtype=torch.float32),
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=CodeDataset(val_df["func"], val_df["cwe_label"]),
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)

    history = pd.DataFrame(trainer.state.log_history).groupby("epoch").first().reset_index()
    return trainer, history


def predict_proba(trainer, texts):
    logits = trainer.predict(CodeDataset(texts)).predictions
    return torch.softmax(torch.tensor(logits), dim=1).numpy()


all_scores = []

## 6. Truncation rate

CodeBERT reads at most 512 tokens, so we cut longer functions at the end. This cell counts how many functions are longer than 512 tokens, for each split and for each class. It uses only the tokenizer, so it is fast.

In [ ]:
def token_count(texts):
    return np.array([len(ids) for ids in tokenizer(list(texts), truncation=False)["input_ids"]])


truncation_rows = []
for name, df in splits.items():
    df["n_tokens"] = token_count(df["func"])
    df["truncated"] = df["n_tokens"] > MAX_LENGTH
    truncation_rows.append({"split": name, "functions": len(df), "truncated": int(df["truncated"].sum()), "share_truncated": df["truncated"].mean(), "median_tokens": df["n_tokens"].median()})

truncation = pd.DataFrame(truncation_rows)
truncation.to_csv(OUT_DIR / "truncation_by_split.csv", index=False)
print(truncation.to_string(index=False))

all_kept = pd.concat(splits.values())
truncation_by_class = all_kept.groupby("cwe_label").agg(functions=("truncated", "size"), share_truncated=("truncated", "mean")).reset_index()
truncation_by_class["train_count"] = truncation_by_class["cwe_label"].map(train_counts)
truncation_by_class["group"] = np.where(truncation_by_class["cwe_label"].isin(rare_classes), "rare", "common")
truncation_by_class.to_csv(OUT_DIR / "truncation_by_class.csv", index=False)
print("\nAll splits together:", round(all_kept["truncated"].mean(), 3))
print(all_kept.assign(group=np.where(all_kept["cwe_label"].isin(rare_classes), "rare", "common")).groupby("group")["truncated"].mean().round(3))

## 7. Simple baselines

**Majority class.** This model always predicts the most common class in train. It shows the lowest score that any useful model should beat.

**TF-IDF with logistic regression.** TF-IDF turns each function into a vector of word counts, where words that appear in many functions get a lower weight. Here a "word" is a name (such as `memcpy`), a number or a single symbol. We use single words and pairs of words that appear in at least 2 training functions, and at most 50,000 of them. The logistic regression uses class weights, like our CodeBERT models. We fixed these settings before we ran the notebook and do not tune them. TF-IDF reads the whole function, so it is not limited to 512 tokens.

In [ ]:
majority = DummyClassifier(strategy="most_frequent").fit(train_df[["func"]], train_df["cwe_label"])
print("Majority class:", majority.predict(test_df[["func"]].iloc[:1])[0])
for name, df in splits.items():
    all_scores.append(scores("majority_class", name, df["cwe_label"], majority.predict(df[["func"]])))

tfidf = TfidfVectorizer(token_pattern=r"[A-Za-z_][A-Za-z0-9_]*|\d+|[^\sA-Za-z0-9_]", lowercase=False, ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df["func"])
tfidf_lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
tfidf_lr.fit(X_train_tfidf, train_df["cwe_label"])
for name, df in splits.items():
    pred = tfidf_lr.predict(tfidf.transform(df["func"]))
    all_scores.append(scores("tfidf_logreg", name, df["cwe_label"], pred))
    if name == "test":
        per_class_f1(list(df["cwe_label"]), list(pred)).to_csv(OUT_DIR / "tfidf_logreg_test_per_class.csv", index=False)

print(pd.DataFrame(all_scores).round(3).to_string(index=False))

## 8. Frozen CodeBERT with logistic regression (Presentation 1 setup)

This is the test the professor asked for. CodeBERT is not trained here ("frozen"). We only use it to turn each function into one vector: the output for the first token, with at most 512 tokens. This is the same embedding as in Presentation 1. Then a logistic regression learns the classes from these vectors, with the same settings as in Presentation 1: `max_iter=1000` and class weights.

We fit it in two ways, on the same 82 classes:

- **Project split:** train on train, test on test. This number decides the diagnosis.
- **Random split:** we mix the functions of train, val and test and split them 80/20 at random (`random_state=42`, as in Presentation 1). Functions from the same project can then be in train and in test. If this score is much higher than the project-split score, leakage between projects explains much of the gap. Note: the Presentation 1 random-split run (0.22) used 139 classes and no class weights, so its number is not exactly comparable.

The embeddings are saved, so this step runs only once. It takes a few minutes on a GPU.

In [ ]:
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()


def cls_embeddings(texts, batch_size=64):
    texts = list(texts)
    out = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = tokenizer(texts[start:start + batch_size], truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(DEVICE)
            out.append(encoder(**batch).last_hidden_state[:, 0].float().cpu().numpy())
    return np.vstack(out)


embeddings = {}
for name, df in splits.items():
    path = OUT_DIR / f"codebert_cls_{name}.npy"
    if not path.exists():
        np.save(path, cls_embeddings(df["func"]))
    embeddings[name] = np.load(path)
    print(name, embeddings[name].shape)

del encoder
if DEVICE == "cuda":
    torch.cuda.empty_cache()

In [ ]:
frozen_lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
frozen_lr.fit(embeddings["train"], train_df["cwe_label"])
for name, df in splits.items():
    pred = frozen_lr.predict(embeddings[name])
    all_scores.append(scores("frozen_codebert_logreg", name, df["cwe_label"], pred))
    if name == "test":
        per_class_f1(list(df["cwe_label"]), list(pred)).to_csv(OUT_DIR / "frozen_codebert_logreg_test_per_class.csv", index=False)

# Random split check on the same functions and classes
X_all = np.vstack([embeddings[name] for name in splits])
y_all = pd.concat([df["cwe_label"] for df in splits.values()]).to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
random_lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(X_tr, y_tr)
all_scores.append(scores("frozen_codebert_logreg_random_split", "train", y_tr, random_lr.predict(X_tr)))
all_scores.append(scores("frozen_codebert_logreg_random_split", "test", y_te, random_lr.predict(X_te)))

print(pd.DataFrame(all_scores).round(3).to_string(index=False))

## 9. Reference model trained again (all labels, seed 0)

This cell trains the reference model of the paper again, with the same code, the same settings and the same seed. In the paper, this model reached 0.031 test macro-F1. We did not save that model, so we train it again. The result may differ a little, because GPU training is not fully deterministic.

This time we save:

- the training history (`reference_history.csv`), with the loss and the validation macro-F1 after each epoch;
- one prediction per function for train, val and test, with the confidence (`reference_<split>_predictions.csv`). The test file also holds the code, the project and the commit;
- the model itself, in `models/diagnosis_reference/` (about 0.5 GB). It is not in the zip file.

The training scores answer the professor's question about training-set accuracy. If the model scores high on train but low on test, it learns the training functions but does not generalize to new projects. If it also scores low on train, the model does not learn at all, and the pipeline is the more likely problem.

This cell takes about 30 minutes on an RTX 4090.

In [ ]:
REF_SEED = 0
REF_TEST_PATH = OUT_DIR / "reference_test_predictions.csv"

if REF_TEST_PATH.exists():
    print("Reference model already done")
else:
    ref_trainer, ref_history = train_model(train_df["func"], train_df["cwe_label"], REF_SEED)
    ref_history.to_csv(OUT_DIR / "reference_history.csv", index=False)
    ref_trainer.save_model(MODEL_OUT)
    for name, df in splits.items():
        proba = predict_proba(ref_trainer, df["func"])
        out = pd.DataFrame({"true": df["cwe_label"], "pred": np.array(classes)[proba.argmax(axis=1)], "confidence": proba.max(axis=1), "n_tokens": df["n_tokens"]})
        if name == "test":
            out = pd.concat([df[["project", "commit_id", "func"]].reset_index(drop=True), out.reset_index(drop=True)], axis=1)
        out.to_csv(OUT_DIR / f"reference_{name}_predictions.csv", index=False)
    del ref_trainer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

ref_preds = {name: pd.read_csv(OUT_DIR / f"reference_{name}_predictions.csv") for name in splits}
for name, df in ref_preds.items():
    all_scores.append(scores("finetuned_codebert_reference_rerun", name, df["true"], df["pred"]))
    if name == "test":
        per_class_f1(list(df["true"]), list(df["pred"])).to_csv(OUT_DIR / "reference_test_per_class.csv", index=False)

print(pd.read_csv(OUT_DIR / "reference_history.csv").filter(items=["epoch", "loss", "eval_loss", "eval_macro_f1"]).round(4).to_string(index=False))
print("\nTest macro-F1 in the paper: 0.031")
print(pd.DataFrame(all_scores).query("model == 'finetuned_codebert_reference_rerun'").round(3).to_string(index=False))

## 10. Confusion matrix

A confusion matrix shows, for each true class, which classes the model predicts. We use the predictions of the reference model on test. The figure shows the 15 classes with the most test functions, and each row shows the share of that class's functions that go to each predicted class. Predictions outside these 15 classes go to the column "other". We also save a table with the 20 most frequent (true, predicted) mistakes over all classes.

In [ ]:
test_pred = ref_preds["test"]
top = test_pred["true"].value_counts().head(15).index.tolist()
shown = test_pred[test_pred["true"].isin(top)]
pred_col = shown["pred"].where(shown["pred"].isin(top), "other")
cm = confusion_matrix(shown["true"], pred_col, labels=top + ["other"])[: len(top)]
cm_share = cm / cm.sum(axis=1, keepdims=True)
pd.DataFrame(cm, index=top, columns=top + ["other"]).to_csv(OUT_DIR / "reference_confusion_top15.csv")

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(cm_share, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(top) + 1), top + ["other"], rotation=90, fontsize=8)
ax.set_yticks(range(len(top)), [f"{c} (n={int(cm[i].sum())})" for i, c in enumerate(top)], fontsize=8)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
fig.colorbar(im, ax=ax, label="Share of true class")
fig.tight_layout()
fig.savefig(FIG_DIR / "reference_confusion_top15.pdf")
plt.show()

mistakes = test_pred[test_pred["true"] != test_pred["pred"]]
top_pairs = mistakes.groupby(["true", "pred"]).size().rename("count").sort_values(ascending=False).head(20).reset_index()
top_pairs.to_csv(OUT_DIR / "reference_top_confusions.csv", index=False)
print(top_pairs.to_string(index=False))

## 11. Three misclassified functions

The professor asked us to explain three misclassified functions. To avoid picking easy examples, we choose them at random from all misclassified test functions, with a fixed seed (`random_state=0`). The cell saves them with their code in `failure_examples.csv`. We write the explanation by hand later: what the code does, the true and predicted CWE, and whether the error makes sense, is a hierarchy issue or looks like a label problem.

In [ ]:
examples = mistakes.sample(3, random_state=0)
examples.to_csv(OUT_DIR / "failure_examples.csv", index=False)
for _, row in examples.iterrows():
    print("=" * 80)
    print(f"Project: {row['project']} | commit: {row['commit_id']}")
    print(f"True: {row['true']} | predicted: {row['pred']} (confidence {row['confidence']:.2f}) | tokens: {row['n_tokens']}")
    print("-" * 80)
    print("\n".join(str(row["func"]).splitlines()[:60]))

## 12. Summary and decision

This cell puts all scores in one table and saves it as `diagnosis_summary.csv`. Then it applies the decision rule from `results.md` to the test macro-F1 of frozen CodeBERT with logistic regression on the project split. At the end, it puts all result files and figures in `results/diagnosis.zip`.

In [ ]:
summary = pd.DataFrame(all_scores).drop_duplicates(["model", "split"], keep="last")
summary.to_csv(OUT_DIR / "diagnosis_summary.csv", index=False)
print(summary.pivot(index="model", columns="split", values="macro_f1").round(3))
print()
print(summary[summary["split"] == "test"].round(3).to_string(index=False))

frozen_test = summary.query("model == 'frozen_codebert_logreg' and split == 'test'")["macro_f1"].iloc[0]
if frozen_test <= 0.05:
    verdict = "0.05 or lower: the project split explains the low scores."
elif frozen_test >= 0.10:
    verdict = "0.10 or higher: the fine-tuning pipeline is probably broken."
else:
    verdict = "between 0.05 and 0.10: unclear."
print(f"\nFrozen CodeBERT + logistic regression, project split, test macro-F1 = {frozen_test:.3f}")
print("Decision:", verdict)

shutil.copytree(FIG_DIR, OUT_DIR / "figures", dirs_exist_ok=True)
archive = shutil.make_archive(str(RESULTS_DIR / "diagnosis"), "zip", OUT_DIR)
print("Download:", archive)